In [1]:
!pip install pandas numpy scikit-learn scipy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\pcoda\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import re
import ast

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

# Preprocessing 

In [3]:
def preprocess(df):
    df = df.copy()

    df = df[[col for col in df.columns if "Unnamed" not in col]]

    # Rating
    def extract_rating(x):
        if pd.isna(x):
            return np.nan
        match = re.search(r'(\d+\.?\d*)', str(x))
        return float(match.group(1)) if match else np.nan

    df["Rating"] = df["Rating"].fillna(df["Review_Score"].apply(extract_rating))
    df["Rating"] = df["Rating"].fillna(df["Rating"].median())

    # Price
    df["Price"] = df.groupby("City")["Price"].transform(
        lambda x: x.fillna(x.median())
    )
    df["Price"] = df["Price"].fillna(df["Price"].median())

    cap = df["Price"].quantile(0.99)
    df["Price_clipped"] = df["Price"].clip(upper=cap)

    # Review count
    def extract_count(x):
        if pd.isna(x):
            return 0
        match = re.search(r'(\d+)', str(x))
        return int(match.group(1)) if match else 0

    df["review_count"] = df["Total Rating"].apply(extract_count)

    # Amenities
    def clean_amenities(x):
        try:
            items = ast.literal_eval(x)
            return " ".join([
                re.sub(r"\(.*?\)", "", i).strip().lower().replace(" ", "_")
                for i in items
            ])
        except:
            return ""

    df["amenity_text"] = df["Amenities"].apply(clean_amenities)
    df["content"] = df["amenity_text"] + " " + df["City"].str.lower()

    return df

### Content based RS, KNN based RS and Hybrid (KNN + Content)

In [4]:
class ContentBasedRecommender:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1,2))

    def fit(self, df):
        self.df = df
        self.matrix = self.vectorizer.fit_transform(df["content"])
        return self

    def recommend(self, idx, top_k=10):
        sim = cosine_similarity(self.matrix[idx], self.matrix).flatten()
        sim[idx] = -1
        idxs = sim.argsort()[::-1][:top_k]

        result = self.df.iloc[idxs].copy()
        result["score"] = sim[idxs]
        return result


class KNNRecommender:
    def fit(self, df):
        self.df = df

        features = df[["Rating","Price_clipped","Latitude","Longitude","review_count"]]
        self.scaler = MinMaxScaler()
        self.X = self.scaler.fit_transform(features)

        self.knn = NearestNeighbors(n_neighbors=21)
        self.knn.fit(self.X)
        return self

    def recommend(self, idx, top_k=10):
        dist, ind = self.knn.kneighbors([self.X[idx]])

        dist = dist.flatten()
        ind = ind.flatten()

        mask = ind != idx
        ind = ind[mask][:top_k]
        dist = dist[mask][:top_k]

        result = self.df.iloc[ind].copy()
        result["score"] = dist
        return result

class HybridRecommender:
    def __init__(self, alpha=0.55):
        self.alpha = alpha
        self.cb = ContentBasedRecommender()
        self.knn = KNNRecommender()

    def fit(self, df):
        self.df = df
        self.cb.fit(df)
        self.knn.fit(df)
        return self

    def recommend(self, idx, top_k=10):
        n = len(self.df)

        # ---------- Content Scores (FULL VECTOR) ----------
        cb_scores = cosine_similarity(
            self.cb.matrix[idx], self.cb.matrix
        ).flatten()

        # ---------- KNN Scores (SPARSE â†’ FULL VECTOR) ----------
        dist, ind = self.knn.knn.kneighbors([self.knn.X[idx]])

        dist = dist.flatten()
        ind = ind.flatten()

        knn_sim_full = np.zeros(n)

        # convert distance â†’ similarity
        for i, d in zip(ind, dist):
            if i != idx:
                knn_sim_full[i] = 1 / (1 + d)

        # ---------- HYBRID ----------
        score = self.alpha * cb_scores + (1 - self.alpha) * knn_sim_full

        score[idx] = -1  # remove self

        idxs = score.argsort()[::-1][:top_k]

        result = self.df.iloc[idxs].copy()
        result["score"] = score[idxs]

        return result

In [5]:
df = pd.read_csv("C:\\Users\\pcoda\\OneDrive\\Dataset{Manually}\\HotelData.csv")

df = preprocess(df)

cb = ContentBasedRecommender().fit(df)
knn = KNNRecommender().fit(df)
hybrid = HybridRecommender().fit(df)

print("âœ… Models ready!")

âœ… Models ready!


# User Input

In [6]:
def get_recommendations_ui():
    hotel_name = input("Enter Hotel Name: ")
    city = input("Enter City (optional, press enter to skip): ")
    algo = input("Algorithm (content/knn/hybrid) [default=content]: ")

    if algo == "":
        algo = "content"

    mask = df["Name"].str.contains(hotel_name, case=False, na=False)

    if city:
        mask &= df["City"].str.lower() == city.lower()

    matches = df[mask]

    if len(matches) == 0:
        print("âŒ Hotel not found")
        return

    idx = matches.index[0]

    if algo == "content":
        recs = cb.recommend(idx)
    elif algo == "knn":
        recs = knn.recommend(idx)
    else:
        recs = hybrid.recommend(idx)

    print("\nðŸ”¥ Top Recommendations:\n")
    display(recs[["Name","City","Rating","Price","score"]])

In [7]:
get_recommendations_ui()


ðŸ”¥ Top Recommendations:



,Name,City,Rating,Price,score
341,Roman Frost,Mahabaleshwar,4.1,5074.0,0.543253
196,Hotel Kohinoor,Mahabaleshwar,3.3,844.0,0.518137
3321,Hotel Grand,Mumbai,4.1,3784.0,0.468881
303,G.S. Residency,Mahabaleshwar,4.6,3487.0,0.459884
332,Prajakta Holiday Inn,Mahabaleshwar,5.0,1653.0,0.459286
287,"Western Valley Cottages , Panchagani",Mahabaleshwar,4.1,2030.0,0.457375
128,Hotel Panchgani Crown,Mahabaleshwar,3.6,2083.0,0.445719
383,Shiv Laxmi Guest House (Super Deluxe Room),Mahabaleshwar,5.0,2160.0,0.443293
180,ShreeKrupa Hotel,Mahabaleshwar,5.0,2271.0,0.420538
367,Shiv Muktai,Mahabaleshwar,4.1,1439.0,0.412216


# Location based RS

## Geo Location-Based RS (Latitude/Longitude)

This recommends hotels that are physically closest to:
1. A user-provided latitude/longitude, or
2. A selected hotel (find nearby alternatives)

It requires coordinate columns in your dataset (preferred: `Latitude` and `Longitude`).

In [9]:
import numpy as np
import pandas as pd

def _pick_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

lat_col = _pick_col(df.columns, ['Latitude', 'latitude', 'Lat', 'lat'])
lon_col = _pick_col(df.columns, ['Longitude', 'longitude', 'Lon', 'lon', 'lng', 'Lng'])

if lat_col is None or lon_col is None:
    print("⚠️  No latitude/longitude columns found in df.")
    print("Add columns named 'Latitude' and 'Longitude' (recommended) to use geo location-based recommendations.")
else:
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')


def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance (km). lat2/lon2 can be numpy arrays."""
    R = 6371.0088
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * (np.sin(dlon / 2.0) ** 2)
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c


def recommend_nearby_hotels(user_lat, user_lon, top_k=10, max_km=None, city=None):
    """Nearest hotels to a user's location."""
    if lat_col is None or lon_col is None:
        raise ValueError("Missing latitude/longitude columns in df")

    candidates = df.dropna(subset=[lat_col, lon_col]).copy()
    if candidates.empty:
        raise ValueError("No rows with valid latitude/longitude")

    if city and 'City' in candidates.columns:
        candidates = candidates[candidates['City'].astype(str).str.lower() == city.lower()]

    dist = haversine_km(user_lat, user_lon, candidates[lat_col].to_numpy(), candidates[lon_col].to_numpy())
    candidates['distance_km'] = dist

    if max_km is not None:
        candidates = candidates[candidates['distance_km'] <= float(max_km)]

    candidates = candidates.sort_values('distance_km').head(int(top_k))
    out_cols = [c for c in ['Name', 'City', 'Rating', 'Price', 'distance_km'] if c in candidates.columns]
    return candidates[out_cols]


def recommend_near_a_hotel(hotel_name, top_k=10, max_km=None):
    """Nearest hotels to a selected hotel (nearby alternatives)."""
    if lat_col is None or lon_col is None:
        raise ValueError("Missing latitude/longitude columns in df")

    matches = df[df['Name'].astype(str).str.contains(str(hotel_name), case=False, na=False)]
    if matches.empty:
        raise ValueError("Hotel not found")

    base = matches.iloc[0]
    base_lat = float(base[lat_col])
    base_lon = float(base[lon_col])

    candidates = df.dropna(subset=[lat_col, lon_col]).copy()
    dist = haversine_km(base_lat, base_lon, candidates[lat_col].to_numpy(), candidates[lon_col].to_numpy())
    candidates['distance_km'] = dist

    # Exclude the same row if possible
    candidates = candidates.drop(index=[matches.index[0]], errors='ignore')

    if max_km is not None:
        candidates = candidates[candidates['distance_km'] <= float(max_km)]

    candidates = candidates.sort_values('distance_km').head(int(top_k))
    out_cols = [c for c in ['Name', 'City', 'Rating', 'Price', 'distance_km'] if c in candidates.columns]
    return candidates[out_cols]


def geo_location_ui():
    if lat_col is None or lon_col is None:
        return

    mode = input("Mode (1=user location, 2=near a hotel) [default=1]: ").strip() or "1"

    try:
        top_k = int(input("Top K [default=10]: ").strip() or "10")
    except Exception:
        top_k = 10

    max_km_in = input("Max distance km (optional): ").strip()
    max_km = float(max_km_in) if max_km_in else None

    if mode == "2":
        hotel_name = input("Enter Hotel Name: ")
        recs = recommend_near_a_hotel(hotel_name, top_k=top_k, max_km=max_km)
    else:
        user_lat = float(input("Enter your latitude: "))
        user_lon = float(input("Enter your longitude: "))
        city = input("City filter (optional): ").strip()
        city = city if city else None
        recs = recommend_nearby_hotels(user_lat, user_lon, top_k=top_k, max_km=max_km, city=city)

    if 'display' in globals():
        display(recs)
    else:
        print(recs)

# Run UI:
geo_location_ui()

                          Name           City  Rating    Price  distance_km
8762               The Carlton     Kodaikanal     4.7  12695.0   202.792600
8830   Hotel Micro Continental  Visakhapatnam     3.8   1865.0   336.160291
2604        Hotel Platinum Inn         Mumbai     2.9   1085.0   532.556428
7284  The Narayani Continental        Gangtok     4.1   1334.0   622.445923
3524           Accord HighLand           Ooty     4.5   7440.0   642.588346
3042        Hotel Sea Princess         Mumbai     4.1   5859.0   655.339268
6101              Boho's Hotel        Varkala     4.4   2072.0   787.540273
3998             Signature Inn    Pondicherry     4.1   1580.0   950.065152
600        Winterfell The Stay         Manali     4.5   2148.0  1305.697930
5064             Jivanta Hotel         Shirdi     4.2   2721.0  1308.503605


In [10]:
def city_wise_recommendation_advanced():
    city = input("Enter City: ")

    city_df = df[df["City"].str.lower() == city.lower()]

    if len(city_df) == 0:
        print("âŒ City not found")
        return

    # Weighted score (rating + review_count)
    city_df = city_df.copy()
    city_df["score"] = city_df["Rating"] * 0.7 + np.log1p(city_df["review_count"]) * 0.3

    top_hotels = city_df.sort_values(by="score", ascending=False).head(5)

    print(f"\nðŸ¨ Top 5 Hotels in {city.title()} (Smart Ranking):\n")
    display(top_hotels[["Name", "City", "Rating", "Price", "review_count"]])

city_wise_recommendation_advanced()


ðŸ¨ Top 5 Hotels in Udaipur (Smart Ranking):



,Name,City,Rating,Price,review_count
10473,Lakshmi Nanda Resort,udaipur,4.1,5051.0,0
9745,HISTORIA ROYAL,udaipur,4.1,2565.0,0
9746,SWAROOP VILAS - LAKE FACING BOUTIQUE HOTEL,udaipur,4.1,2336.0,0
9747,Uddhav Vilas,udaipur,4.1,1078.0,0
9748,Hotel Meenakshi Udaipur,udaipur,4.1,1233.0,0


In [11]:
def budget_based_recommendation():
    min_price = float(input("Enter Minimum Price (â‚¹): "))
    max_price = float(input("Enter Maximum Price (â‚¹): "))
    city = input("Enter City (optional): ")

    filtered = df[
        (df["Price"] >= min_price) &
        (df["Price"] <= max_price)
    ]

    if city:
        filtered = filtered[filtered["City"].str.lower() == city.lower()]

    if len(filtered) == 0:
        print("âŒ No hotels found in this budget")
        return

    # Sort by rating
    filtered = filtered.sort_values(by="Rating", ascending=False).head(10)

    print("\nðŸ’° Best Hotels in Your Budget:\n")
    display(filtered[["Name", "City", "Rating", "Price"]])

budget_based_recommendation()


ðŸ’° Best Hotels in Your Budget:



,Name,City,Rating,Price
3516,Greenyards,Ooty,5.0,1433.0
3602,Rose Park Residency,Ooty,5.0,1991.0
3595,MANGO HILL CENTRAL OOTY,Ooty,5.0,1985.0
3621,OYO Flagship 44245 Hotel Rainforest Thalayathi...,Ooty,5.0,1162.0
3589,Lawley Cottage,Ooty,5.0,1259.0
3517,Aakash Rooms and cottages,Ooty,4.5,1387.0
3487,Zostel Ooty,Ooty,4.5,1098.0
3511,RK Homes Guest House,Ooty,4.3,1883.0
3577,Glen Park Inn,Ooty,4.2,1567.0
3556,Osprey Cottage,Ooty,4.1,1999.0


In [ ]:
def best_value_hotels():
    city = input("Enter City (optional): ")

    data = df.copy()

    if city:
        data = data[data["City"].str.lower() == city.lower()]

    if len(data) == 0:
        print("âŒ No hotels found")
        return

    # Value Score Formula
    data["value_score"] = (
        data["Rating"] * 0.6 +
        np.log1p(data["review_count"]) * 0.3 -
        (data["Price"] / data["Price"].max()) * 0.1
    )

    top_hotels = data.sort_values(by="value_score", ascending=False).head(10)

    print("\nðŸ’Ž Best Value for Money Hotels:\n")
    display(top_hotels[["Name", "City", "Rating", "Price" , "Amenities"]])

best_value_hotels()


ðŸ’Ž Best Value for Money Hotels:



,Name,City,Rating,Price,Amenities
3514,Hotel Lakeview,Ooty,5.0,3029.0,"['Gloves (Free)', 'Sanitizers', 'Sanitizers in..."
3486,Garden Manor,Ooty,5.0,2308.0,"['Masks (Free)', 'Gloves (Free)', 'Sanitizers ..."
3515,Sinclairs Retreat Ooty,Ooty,5.0,7161.0,"['Sanitizers', 'Disinfection', 'Contactless ch..."
3570,OYO 10950 Hotel Hills Palace,Ooty,5.0,3288.0,"['Sanitizers', 'Face shields (for hotel staff)..."
3497,DARSHAN HOTEL,Ooty,4.2,4840.0,"['Sanitizers', 'Thermal screening at entry and..."
3602,Rose Park Residency,Ooty,5.0,1991.0,"['Masks (Free)', 'Gloves (Paid)', 'Sanitizers ..."
3516,Greenyards,Ooty,5.0,1433.0,"['Kitchen/Kitchenette', 'Power backup', 'House..."
3631,Sterling Ooty Elk Hill,Ooty,4.4,8400.0,"['Masks (Paid)', 'Sanitizers (Paid)', 'Face sh..."
3500,Hotel Sun Park,Ooty,5.0,3800.0,"['Sanitizers installed', 'Parking (Free)', 'Ro..."
3554,Hotel Mount n Mist,Ooty,5.0,4195.0,"['Sanitizers', 'Parking (Free)', 'Wifi (Free)'..."
